# CSCIE89 : Neural Networks and CNNs on MNIST (torchvision)

This notebook repeats the full sequence of our earlier examples (vanilla network, basic CNN, multi-class CNN with model selection, and filter visualization) with one change: the data is the **MNIST** dataset from `torchvision.datasets`, not scikit-learn's 8×8 `digits` or the Flower Photos folders. MNIST images are **28 × 28** grayscale handwritten digits from 0 to 9.

Because MNIST is a torchvision **Dataset**, we pass it straight to a `DataLoader`. We do not need `TensorDataset` or a custom `Dataset` class.

Our Steps:

        Part A - Vanilla network (no CNN)
                load images, split data, build model, train model, test model, predict one image
        Part B - CNN
                build a CNN, train with validation and keep the best epoch, evaluate on the test set,
                inspect predictions and mistakes, save and reload
        Part C - Look inside the CNN
                filters, feature maps, prediction, Grad-CAM heatmap

> **Train/valid/test.** MNIST ships with 60,000 training images and 10,000 test images. We keep the 10,000 test images untouched, and split the 60,000 into training and validation sets. The training set teaches the model, the validation set helps us observe learning and pick the best epoch, and the test set gives the final estimate of performance.

The first run needs internet access to download MNIST (about 10 MB). Downloads are reused on subsequent runs.

## 1. Setup and imports

If an import fails, uncomment the installation line, run it once, then restart the kernel. Ask for help on Ed discussions or email the cscie89 teaching staff.

In [1]:
# %pip install torch torchvision scikit-learn matplotlib pillow

from pathlib import Path
import copy
import json
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import datasets, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay,
)

# Seeds make the split and initial weights repeatable, but do not guarantee
# identical floating-point results across all devices or PyTorch versions.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Apple Silicon use MPS; laptops with NVIDIA GPUs can use CUDA.
# MNIST is small enough to train on a CPU, just slower.
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')

IMAGE_SIZE = 28
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 0.001
DATA_DIR = Path('./data')
OUTPUT_DIR = Path('outputs') / 'mnist'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PyTorch:', torch.__version__, '| Torchvision:', torchvision.__version__, '| Device:', device)

PyTorch: 2.14.1+cu130 | Torchvision: 0.29.1+cu130 | Device: cpu


## 2. Load MNIST with torchvision and look at the images

`datasets.MNIST` downloads the files into `DATA_DIR` the first time and reuses them afterwards. `train=True` selects the 60,000 training images and `train=False` the 10,000 test images.

We first load **without a transform**. The `.data` attribute then holds the raw `uint8` pixels (0 = black, 255 = white) and `.targets` holds one integer label per image. Each image is a 28×28 grid, so the first dimension counts images: `(60000, 28, 28)`.

The class names come from the dataset itself (`.classes`), so we read the label mapping instead of guessing it.

In [2]:
raw_train = datasets.MNIST(root=DATA_DIR, train=True, download=True)
raw_test = datasets.MNIST(root=DATA_DIR, train=False, download=True)
class_names = raw_train.classes

print('Training images:', tuple(raw_train.data.shape), raw_train.data.dtype)
print('Test images:    ', tuple(raw_test.data.shape))
print('Labels:         ', tuple(raw_train.targets.shape), raw_train.targets.dtype)
print('Pixel range:', raw_train.data.min().item(), 'to', raw_train.data.max().item())
print('Classes:', class_names)
print('Class-to-label mapping:', raw_train.class_to_idx)

# Display one example from each class.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.flat):
    example_index = (raw_train.targets == digit).nonzero()[0].item()
    ax.imshow(raw_train.data[example_index], cmap='gray')
    ax.set_title(f'Label: {digit}')
    ax.axis('off')
plt.suptitle('One image from each class')
plt.tight_layout()
plt.show()

Training images: (60000, 28, 28) torch.uint8
Test images:     (10000, 28, 28)
Labels:          (60000,) torch.int64
Pixel range: 0 to 255
Classes: ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']
Class-to-label mapping: {'0 - zero': 0, '1 - one': 1, '2 - two': 2, '3 - three': 3, '4 - four': 4, '5 - five': 5, '6 - six': 6, '7 - seven': 7, '8 - eight': 8, '9 - nine': 9}


## 3. Create training, validation, and test datasets

The test set is already separated for us. We split the 60,000 training images into **55,000 for training and 5,000 for validation**. `stratify` keeps similar proportions of each digit in both groups. We split *indices*, then use `Subset` to give each split a view of the same underlying dataset.

`ToTensor()` converts a PIL image with integer pixels 0–255 into a float tensor with values 0–1, and arranges the dimensions as **channels, height, width**. MNIST is grayscale, so it has one channel: `(1, 28, 28)`. This is the same 0–1 scaling we applied to `digits` by dividing by 16. Every split uses the same deterministic preprocessing.

In [3]:
image_transform = transforms.ToTensor()   # (28, 28) uint8 -> (1, 28, 28) float in [0, 1]

full_train = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=image_transform)
test_dataset = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=image_transform)

train_indices, val_indices = train_test_split(
    np.arange(len(full_train)), test_size=5_000, random_state=SEED,
    stratify=full_train.targets.numpy(),
)
train_dataset = Subset(full_train, train_indices)
val_dataset = Subset(full_train, val_indices)

# An index must appear in only one split: this checks for accidental overlap.
assert set(train_indices).isdisjoint(val_indices)

total = len(full_train) + len(test_dataset)
for name, data in [('Training', train_dataset), ('Validation', val_dataset), ('Test', test_dataset)]:
    print(f'{name:10s}: {len(data):5d} images ({len(data) / total:.1%} of 70,000)')

# A Dataset returns one (image, label) pair at a time.
image, label = train_dataset[0]
print('One image tensor:', tuple(image.shape), image.dtype, '| label:', label)
print(f'Pixel range after ToTensor: {image.min():.1f} to {image.max():.1f}')

Training  : 55000 images (78.6% of 70,000)
Validation:  5000 images (7.1% of 70,000)
Test      : 10000 images (14.3% of 70,000)
One image tensor: (1, 28, 28) torch.float32 | label: 6
Pixel range after ToTensor: 0.0 to 1.0


## 4. Put the data into DataLoaders

`DataLoader` collects images from a Dataset into **batches** of 64. Training batches are shuffled each epoch so the model does not learn from a fixed ordering. Validation and test data do not need shuffling.

`train_generator` creates a random-number generator with a fixed starting point. In our case it controls how the DataLoader shuffles the training images.

In [4]:
# The generator makes the shuffled training order reproducible.
train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=train_generator, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

batch_images, batch_labels = next(iter(train_loader))
print('Image batch shape:', tuple(batch_images.shape))
print('Label batch shape:', tuple(batch_labels.shape), batch_labels.dtype)
assert batch_images.shape[1:] == (1, IMAGE_SIZE, IMAGE_SIZE)

Image batch shape: (64, 1, 28, 28)
Label batch shape: (64,) torch.int64


# Part A: A vanilla network (this does NOT use a CNN)

We start with the smallest complete program: a fully connected network. It sees each image as a flat list of 784 numbers and ignores which pixels are neighbors.

## 5. Build one model

`nn.Sequential` connects the layers in order:

- **Flatten:** turns each 28×28 image into a list of 784 numbers, keeping images separate.
- **Linear(784, 64):** learns 64 combinations of those pixel values.
- **ReLU:** replaces negative values with zero, allowing the network to learn nonlinear patterns.
- **Linear(64, 10):** produces one score for each possible digit.

For a batch of images, the shape changes are `(batch, 1, 28, 28) → (batch, 784) → (batch, 64) → (batch, 64) → (batch, 10)`.

The model starts with random weights. We use `Sequential` directly, so a custom class and a separate `forward` method are unnecessary. `CrossEntropyLoss` takes the raw scores (logits), so we do not add softmax. Adam uses the gradients to update the weights.

In [5]:
vanilla_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(IMAGE_SIZE * IMAGE_SIZE, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
).to(device)

vanilla_loss_function = nn.CrossEntropyLoss()
vanilla_optimizer = torch.optim.Adam(vanilla_model.parameters(), lr=LEARNING_RATE)

print(vanilla_model)
print('Trainable parameters:', sum(p.numel() for p in vanilla_model.parameters()))

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=10, bias=True)
)
Trainable parameters: 50890


## 6. Train the vanilla model

An **epoch** is one complete pass through all training images. We make 5 passes. For every batch, we predict the digits, calculate the loss, calculate gradients, and update the weights.

`zero_grad()` clears gradients left over from the previous batch. `backward()` calculates new gradients. **`optimizer.step()` is the line that changes the weights.** The average loss printed below should generally decrease. The validation and test images do not appear in this loop.

In [6]:
vanilla_epochs = 5
vanilla_model.train()   # Put the model in training mode.

for epoch in range(vanilla_epochs):
    total_loss = 0.0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        vanilla_optimizer.zero_grad()                          # Clear old gradients.
        scores = vanilla_model(inputs)                         # Predict this batch.
        loss = vanilla_loss_function(scores, targets)          # Measure the error.
        loss.backward()                                        # Calculate gradients.
        vanilla_optimizer.step()                               # Update the weights.

        # Weight each batch's average loss by the number of images in it.
        total_loss += loss.item() * len(inputs)

    average_loss = total_loss / len(train_dataset)
    print(f'Epoch {epoch + 1:2d}/{vanilla_epochs} | Loss: {average_loss:.4f}')

Epoch  1/5 | Loss: 0.4077


Epoch  2/5 | Loss: 0.1964


Epoch  3/5 | Loss: 0.1453


Epoch  4/5 | Loss: 0.1179


Epoch  5/5 | Loss: 0.0956


## 7. Test the vanilla model

Now we give the model images it did not see during training. Each row of scores has ten entries. `argmax(dim=1)` chooses the index of the largest score in each row: that is the predicted digit.

`model.eval()` selects evaluation behavior. `torch.no_grad()` turns off gradient tracking because we are only checking predictions. Neither line changes the learned weights.

**Accuracy = number of correct predictions ÷ number of test images.** The 10,000 test images are scored in batches, which keeps memory use small.

In [7]:
vanilla_model.eval()
correct = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        test_scores = vanilla_model(inputs.to(device))
        test_predictions = test_scores.argmax(dim=1).cpu()
        correct += (test_predictions == targets).sum().item()

vanilla_accuracy = correct / len(test_dataset)
print(f'Correct predictions: {correct} out of {len(test_dataset)}')
print(f'Test accuracy: {vanilla_accuracy:.2%}')

Correct predictions: 9686 out of 10000
Test accuracy: 96.86%


## 8. Look at one prediction

We display the first test image with its predicted and actual digit. Change `index` to inspect a different example.

In [8]:
index = 0
one_image, actual_digit = test_dataset[index]
with torch.no_grad():
    predicted_digit = vanilla_model(one_image.unsqueeze(0).to(device)).argmax(dim=1).item()

plt.imshow(one_image[0], cmap='gray')
plt.title(f'Predicted: {predicted_digit} | Actual: {actual_digit}')
plt.axis('off')
plt.show()

# Part B: A convolutional neural network (CNN)

A CNN keeps the 2D layout of the image. Convolution filters slide across it, so the same small pattern detector is reused at every position.

## 9. Build a vanilla CNN

A convolution learns small filters that detect patterns such as edges and strokes. ReLU replaces negative values with zero and enables nonlinear relationships. Max pooling halves the spatial dimensions, retaining strong responses. Flatten turns the final feature maps into a vector; linear layers turn that vector into class scores, called logits.

For one image the shapes are: `1×28×28 → 16×14×14 → 32×7×7 → 64×3×3 → 576 → 64 → 10`. Each arrow through the feature extractor includes convolution, ReLU, and pooling. The last pooling turns 7×7 into 3×3 because `MaxPool2d(2)` rounds odd sizes down; the last row and column of the 7×7 map are dropped.

We use cross-entropy loss: it expects integer targets and handles the softmax-related calculation internally, so do not add softmax before this loss. The network has 10 output logits, one per digit.

In [9]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),    # padding=1 keeps height and width
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 28 x 28 -> 14 x 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 14 x 14 -> 7 x 7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 7 x 7 -> 3 x 3 (rounds down)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                  # 64 maps x 3 x 3 = 576 values
            nn.Linear(64 * (IMAGE_SIZE // 8) ** 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = BasicCNN(len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

# Confirm that a batch produces one row of 10 scores per image.
with torch.no_grad():
    assert model(batch_images.to(device)).shape == (len(batch_images), len(class_names))

BasicCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=576, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=10, bias=True)
  )
)
Trainable parameters: 60874


## 10. Define evaluation, then train

One epoch is a complete pass through training data. Each batch follows five steps: clear gradients, predict, calculate loss, backpropagate, update weights. Evaluation uses `model.eval()` and `torch.no_grad()` because it must not update weights or build a gradient graph.

We multiply each batch's mean loss by its actual size before averaging across examples, so a smaller final batch has the correct weight. Training metrics below are measured while weights change; validation metrics use the fixed weights at the end of each epoch.

Logits are the raw scores the model produces for each class. They are not probabilities: they can be negative and do not need to sum to 1. The highest score gives the predicted digit.

We keep the weights with the lowest **validation loss**, not the highest test accuracy. The test loader is not used during training. Never adjust settings based on test results; use validation results for experiments.

In [10]:
def evaluate(model, loader):
    """Return average loss and accuracy without changing model weights."""
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            loss_sum += criterion(logits, labels).item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return loss_sum / count, correct / count

history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}
best_val_loss = float('inf')
best_state = None
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum, correct, count = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()             # 1. Clear gradients from the last batch.
        logits = model(inputs)            # 2. Forward pass.
        loss = criterion(logits, labels)  # 3. Compare scores with true labels.
        loss.backward()                   # 4. Calculate weight gradients.
        optimizer.step()                  # 5. Update weights using Adam.
        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(1) == labels).sum().item()
        count += len(labels)
    train_loss, train_accuracy = loss_sum / count, correct / count
    val_loss, val_accuracy = evaluate(model, val_loader)
    for key, value in [('train_loss', train_loss), ('val_loss', val_loss),
                       ('train_accuracy', train_accuracy), ('val_accuracy', val_accuracy)]:
        history[key].append(value)
    if val_loss < best_val_loss:
        best_val_loss, best_epoch = val_loss, epoch
        # A copy is necessary: a plain reference would keep changing during training.
        best_state = copy.deepcopy(model.state_dict())
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f}, acc {train_accuracy:.2%}'
          f' | val loss {val_loss:.4f}, acc {val_accuracy:.2%}', flush=True)

model.load_state_dict(best_state)
print(f'Restored epoch {best_epoch}, selected using validation loss.')
print(f'Training time: {(time.perf_counter() - start) / 60:.1f} minutes')

Epoch 01/15 | train loss 0.2676, acc 91.68% | val loss 0.0821, acc 97.52%


Epoch 02/15 | train loss 0.0662, acc 97.94% | val loss 0.0549, acc 98.32%


Epoch 03/15 | train loss 0.0447, acc 98.59% | val loss 0.0406, acc 98.64%


Epoch 04/15 | train loss 0.0365, acc 98.87% | val loss 0.0420, acc 98.62%


Epoch 05/15 | train loss 0.0291, acc 99.07% | val loss 0.0491, acc 98.54%


...

Epoch 12/15 | train loss 0.0110, acc 99.63% | val loss 0.0405, acc 99.00%


Epoch 13/15 | train loss 0.0087, acc 99.72% | val loss 0.0313, acc 99.10%


Epoch 14/15 | train loss 0.0087, acc 99.72% | val loss 0.0395, acc 99.00%


Epoch 15/15 | train loss 0.0086, acc 99.74% | val loss 0.0425, acc 98.80%


Restored epoch 11, selected using validation loss.
Training time: 2.3 minutes


## 11. Plot learning curves

Lower loss and higher accuracy generally indicate learning. Training improvement accompanied by worsening validation loss suggests overfitting: the model fits training details that do not generalize. The dashed line marks the selected epoch.

Try `EPOCHS = 50` once and watch the gap between the training and validation curves. That is why we keep an eye on validation loss rather than training loss alone.

In [11]:
epochs = np.arange(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(epochs, history['train_' + metric], marker='o', label='Training')
    ax.plot(epochs, history['val_' + metric], marker='o', label='Validation')
    ax.axvline(best_epoch, color='gray', linestyle='--', label='Selected epoch')
    ax.set(xlabel='Epoch', ylabel=metric.capitalize(), title=metric.capitalize() + ' by epoch')
    ax.legend()
    ax.grid(alpha=0.25)
axes[1].set_ylim(0.9, 1.005)   # MNIST accuracy is high; zoom in so differences are visible.
plt.tight_layout()
plt.show()

## 12. Evaluate the selected model on the test set

**Accuracy** is the fraction correct. **Precision** asks how often predictions of a class are right; **recall** asks how many real members of that class we found. **F1** balances precision and recall. **Support** is the number of examples in a class. Macro averages give each class equal weight; weighted averages give more weight to larger classes.

Balanced accuracy averages recall across classes, which helps when class counts differ (MNIST digits are only roughly balanced). We also compare with a trivial model that always predicts the most common *training* class. Confusion-matrix rows are true classes and columns are predictions; correct predictions are on the main diagonal and off-diagonal cells are mistakes.

In [12]:
model.eval()
true_labels, predicted_labels, probabilities = [], [], []
test_loss_sum = 0.0
with torch.no_grad():
    for inputs, labels in test_loader:
        logits = model(inputs.to(device))
        test_loss_sum += criterion(logits, labels.to(device)).item() * len(labels)
        # Softmax is useful here to display scores as probabilities.
        # These are model scores, not guarantees or calibrated confidence estimates.
        probabilities.extend(logits.softmax(1).cpu().numpy())
        predicted_labels.extend(logits.argmax(1).cpu().numpy())
        true_labels.extend(labels.numpy())
true_labels = np.asarray(true_labels)
predicted_labels = np.asarray(predicted_labels)
probabilities = np.asarray(probabilities)

test_loss = test_loss_sum / len(true_labels)
test_accuracy = accuracy_score(true_labels, predicted_labels)
balanced_accuracy = balanced_accuracy_score(true_labels, predicted_labels)
majority_class = np.bincount(full_train.targets[train_indices].numpy()).argmax()
baseline_accuracy = np.mean(true_labels == majority_class)

print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.2%}')
print(f'Balanced accuracy: {balanced_accuracy:.2%}')
print(f'Always predict "{class_names[majority_class]}" baseline: {baseline_accuracy:.2%}')
print(f'Vanilla (non-CNN) model test accuracy from Part A: {vanilla_accuracy:.2%}')
print(classification_report(true_labels, predicted_labels,
      labels=list(range(len(class_names))), target_names=class_names, zero_division=0, digits=4))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(true_labels, predicted_labels,
    labels=list(range(len(class_names))), display_labels=list(range(10)),
    cmap='Blues', colorbar=False, ax=ax)
ax.set_title('Test confusion matrix')
plt.tight_layout()
plt.show()

Test loss: 0.0312
Test accuracy: 99.05%
Balanced accuracy: 99.04%
Always predict "1 - one" baseline: 11.35%
Vanilla (non-CNN) model test accuracy from Part A: 96.86%
              precision    recall  f1-score   support

    0 - zero     0.9939    0.9939    0.9939       980
     1 - one     0.9956    0.9947    0.9952      1135
     2 - two     0.9961    0.9855    0.9907      1032
   3 - three     0.9825    0.9990    0.9907      1010
    4 - four     0.9919    0.9939    0.9929       982
    5 - five     0.9822    0.9888    0.9855       892
     6 - six     0.9947    0.9885    0.9916       958
   7 - seven     0.9808    0.9951    0.9879      1028
   8 - eight     0.9948    0.9867    0.9907       974
    9 - nine     0.9920    0.9782    0.9850      1009

    accuracy                         0.9905     10000
   macro avg     0.9904    0.9904    0.9904     10000
weighted avg     0.9905    0.9905    0.9905     10000



## 13. Inspect predictions and mistakes

Visual inspection helps reveal ambiguous images and recurring errors. The first grid is a reproducible random test sample; the second contains the first mistakes in test-set order. Correct titles are green and incorrect titles are red.

In [13]:
def show_predictions(indices, title):
    if len(indices) == 0:
        print('No mistakes in this test run.')
        return
    rows = (len(indices) + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(10, 3 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, index in zip(axes.flat, indices):
        image, actual = test_dataset[int(index)]
        predicted = predicted_labels[index]
        ax.imshow(image[0], cmap='gray')   # image is (1, 28, 28); drop the channel axis for plotting.
        ax.set_title(f'True: {actual} | Pred: {predicted}\n'
                     f'Score: {probabilities[index, predicted]:.1%}',
                     color='green' if actual == predicted else 'red', fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

rng = np.random.default_rng(SEED)
show_predictions(rng.choice(len(test_dataset), size=12, replace=False), 'Sample test predictions')
errors = np.flatnonzero(true_labels != predicted_labels)
print(f'Misclassified test images: {len(errors)} of {len(test_dataset)}')
show_predictions(errors[:8], 'Examples the model got wrong')

Misclassified test images: 95 of 10000


## 14. Save, reload, and predict one image

We save the weights together with class names and image size so the label mapping is not lost. Reconstruct the same model architecture before loading weights. The example below checks that reloaded weights give the same scores and demonstrates prediction for a single image. This checkpoint is also what Part C loads.

In [14]:
model_path = OUTPUT_DIR / 'mnist_cnn.pth'
torch.save({
    'state_dict': {key: value.cpu() for key, value in model.state_dict().items()},
    'class_names': class_names, 'image_size': IMAGE_SIZE, 'best_epoch': best_epoch,
}, model_path)

# Store metrics separately in a human-readable file.
metrics = {'test_loss': test_loss, 'test_accuracy': test_accuracy,
           'balanced_accuracy': balanced_accuracy,
           'majority_baseline_accuracy': float(baseline_accuracy),
           'vanilla_test_accuracy': vanilla_accuracy,
           'best_epoch': best_epoch, 'history': history}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))

checkpoint = torch.load(model_path, map_location='cpu', weights_only=True)
assert checkpoint['image_size'] == IMAGE_SIZE
reloaded = BasicCNN(len(checkpoint['class_names'])).to(device)
reloaded.load_state_dict(checkpoint['state_dict'])
reloaded.eval()

one_image, actual = test_dataset[0]
with torch.no_grad():
    batch = one_image.unsqueeze(0).to(device)   # Add batch dimension: (1, 1, 28, 28).
    torch.testing.assert_close(model(batch), reloaded(batch))
    predicted = reloaded(batch).argmax(1).item()
print('Reload verified. Example:', checkpoint['class_names'][predicted],
      '| Actual:', class_names[actual])
print('Saved to:', model_path.resolve())

def predict_image(picture):
    """Predict one 28 x 28 grayscale PIL image."""
    batch = image_transform(picture.convert('L')).unsqueeze(0).to(device)
    with torch.no_grad():
        scores = reloaded(batch).softmax(1)[0].cpu()
    index = scores.argmax().item()
    return checkpoint['class_names'][index], scores[index].item()

# Replace this with Image.open('your_digit.png') to try your own 28 x 28 image
# (white digit on a black background, as in MNIST).
print('Single-image prediction:', predict_image(Image.fromarray(test_dataset.data[0].numpy())))

Reload verified. Example: 7 - seven | Actual: 7 - seven
Saved to: /tmp/claude-0/-home-user-lec03/82af13de-b088-581a-a30e-c84accc06e02/scratchpad/run/outputs/mnist/mnist_cnn.pth
Single-image prediction: ('7 - seven', 1.0)


# Part C: Visualize the filters in our MNIST CNN

We now look inside the network. We do **not** train a new model or update its weights in this part.

What we will visualize:

    a) CNN filters themselves
            - How does a filter look
            - How does a filter look in the deeper layers
    b) Intermediate layers and feature maps created by the filters
    c) Heatmap of class activation (Grad-CAM)

## 15. Load our saved checkpoint and recreate the model

A checkpoint contains the learned weights and information such as class names and image size. It does not contain the Python class that defines the architecture, so we reuse the `BasicCNN` class from Part B.

CPU is sufficient for visualizing one image. `map_location='cpu'` also lets us load weights that were originally trained on a GPU. `weights_only=True` uses PyTorch's restricted checkpoint loader.

Constructing the class initially creates random parameters. **`load_state_dict` immediately replaces them with our MNIST-trained parameters.** `eval()` sets prediction mode; it does not erase the learned weights.

In [15]:
viz_dir = OUTPUT_DIR / 'filter_visualizations'
viz_dir.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(model_path, map_location='cpu', weights_only=True)
class_names = checkpoint['class_names']
IMAGE_SIZE = checkpoint['image_size']

# From here on, `model` is a CPU copy restored from the checkpoint file.
model = BasicCNN(len(class_names))
model.load_state_dict(checkpoint['state_dict'])
model.eval()

print('Our checkpoint:', model_path)
print('Classes:', class_names)
print('Image size:', IMAGE_SIZE)
print(model)

Our checkpoint: outputs/mnist/mnist_cnn.pth
Classes: ['0 - zero', '1 - one', '2 - two', '3 - three', '4 - four', '5 - five', '6 - six', '7 - seven', '8 - eight', '9 - nine']
Image size: 28
BasicCNN(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=576, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=10, bias=True)
  )
)


## 16. Understand a filter's shape

**Filters are learned weights. Feature maps are the results of using those weights on an image.** A filter does not change when we choose a different image; its resulting feature map does.

PyTorch stores convolution weights as **(output channels, input channels, kernel height, kernel width)**.

The first layer's `(16, 1, 3, 3)` tensor means 16 filters, each containing **one** 3×3 slice. Unlike the flower photos (red, green, blue), MNIST has a single grayscale input channel, so every first-layer filter is just one 3×3 grid of numbers. There is no RGB composite to build for it.

In deeper layers, input channels represent learned features rather than pixels. Their weights can also be plotted as individual slices, but not interpreted as ordinary image filters.

In [16]:
for index, layer in enumerate(model.features):
    if isinstance(layer, nn.Conv2d):
        print(f'features[{index}]: weights {tuple(layer.weight.shape)}'
              f', biases {tuple(layer.bias.shape)}')

first_conv = model.features[0]
filters = first_conv.weight.detach().cpu()
print('One complete first-layer filter:', tuple(filters[0].shape))

features[0]: weights (16, 1, 3, 3), biases (16,)
features[3]: weights (32, 16, 3, 3), biases (32,)
features[6]: weights (64, 32, 3, 3), biases (64,)
One complete first-layer filter: (1, 3, 3)


## 17. Plot the first-layer filters

Each panel below is **one filter**: a 3×3 grid of numbers, magnified for visibility. They are not photographs. Blue means a negative weight, white means approximately zero, and red means a positive weight. Every panel uses the **same color scale**, so weight strengths are comparable. We only read the weights; the plotting scale does not modify the model's parameters.

Positive and negative weights contribute differently to a weighted sum. These pictures alone do not prove that a filter detects a specific stroke or digit part; the input values, bias, other channels, and later operations all matter.

In [17]:
limit = max(filters.abs().max().item(), 1e-8)   # Avoid a zero-width scale if all weights are zero.
fig, axes = plt.subplots(4, 4, figsize=(7, 7), layout='constrained')
for filter_index, ax in enumerate(axes.flat):
    image = ax.imshow(filters[filter_index, 0], cmap='RdBu_r',
                      vmin=-limit, vmax=limit, interpolation='nearest')
    ax.set_title(f'Filter {filter_index}')
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(image, ax=axes, shrink=0.6, label='Learned weight value')
fig.suptitle('All 16 first-layer filters (one 3×3 slice each)')
fig.savefig(viz_dir / 'first_layer_filters.png', dpi=150)
plt.show()

print(model.features[0].weight[0].detach())
# One 3×3 matrix inside a (1, 3, 3) tensor: the single grayscale input channel.

tensor([[[ 0.1575,  0.5907,  0.0896],
         [ 0.5424,  0.0791, -0.3842],
         [-0.1529, -0.6373, -0.4728]]])


For one first-layer FILTER:

| Input channel | Learned weights |
|---|---:|
| Grayscale | 3 × 3 = 9 |
| **Total** | **9** |

It also has one bias, stored separately. Our first layer contains 16 such filters.

A "real filter" is exactly these learned numbers, not a hidden photograph.

## 17A. Filter, kernel, channel, feature: what each word means

Different books sometimes use **filter** and **kernel** interchangeably. The most reliable way to resolve the ambiguity is to ask for the tensor shape. In this notebook we use these conventions:

| Term | Meaning here | Concrete example |
|---|---|---|
| Convolutional layer | One operation containing a collection of learned filters | `Conv2d(1, 16, 3)` is **one layer**, not 16 layers |
| Filter | All the weights used to produce **one output channel** | First-layer filter 0 has shape `(1, 3, 3)` and 9 weights, plus one separate bias |
| Kernel size | The height and width of the sliding window | `kernel_size=3` means a 3×3 spatial window; it does not mean three filters |
| Kernel slice | Our name for the 2D weights connecting one input channel to one output channel | `weight[output_channel, input_channel]` is a 3×3 slice |
| Input channel | One input plane to a layer | Block 1 receives the grayscale image; block 2 receives 16 learned response planes |
| Output channel | One response plane produced by one filter | 16 filters produce 16 output channels |
| Feature map | A 2D plane of spatial responses for a particular image | Channel 0's 28×28 map after the first ReLU |
| Feature | An informal name for a learned pattern or representation | A stroke direction or curve; not necessarily a named digit part |
| Activation | A value or tensor produced while processing an input | A ReLU activation is the nonnegative result after applying ReLU |
| Block | A group of operations; its meaning depends on the model | Here, each block is Conv2d → ReLU → MaxPool2d |

**One filter combines channels; it does not produce a separate output map for each input channel.** At one spatial location, a block-2 filter multiplies each of its 16 slices by the matching 3×3 patch of the 16 input maps. It sums those 144 products and adds one bias to make **one output number**. Sliding it across the maps creates one map. Repeating that with all 32 filters creates 32 maps.

For this model (ordinary convolutions with `groups=1`):

| Block | PyTorch weight shape | Complete filters | Slices in each filter | Weights in each filter | Output after ReLU for a 28×28 input |
|---|---|---:|---:|---:|---|
| 1 | `(16, 1, 3, 3)` | 16 | 1 grayscale slice | 9 | `(batch, 16, 28, 28)` |
| 2 | `(32, 16, 3, 3)` | 32 | 16 learned-channel slices | 144 | `(batch, 32, 14, 14)` |
| 3 | `(64, 32, 3, 3)` | 64 | 32 learned-channel slices | 288 | `(batch, 64, 7, 7)` |

Each filter also has one bias, not included in the weight counts above. Pooling changes spatial size but preserves channel count. Thus block 1's 16 output channels become block 2's 16 input channels after pooling. **Channels 0, 1, and 2 in block 2 are not pixels or colors.**

## 18. A readable grid that shows every filter and every input-channel slice

A deep filter has many channel slices. We make a grid in which **each row is one complete filter** and **each column is one input channel**. Every tiny colored tile is that connection's exact 3×3 kernel slice.

This keeps all weights visible without averaging away channel differences. White lines separate the tiles; they are not extra weights. Blue indicates negative weights and red positive weights. Every tile within a block uses one shared scale. Each block has its own colorbar, so check its numbers when comparing blocks. No scientific meaning should be inferred from the arbitrary numeric ordering of learned channels.

In [18]:
def show_all_filter_slices(conv_layer, block_number):
    # The tensor axes are: output filter, input channel, kernel height, kernel width.
    weights = conv_layer.weight.detach().cpu().numpy()
    filter_count, input_channels, kernel_height, kernel_width = weights.shape
    tile_height, tile_width = kernel_height + 1, kernel_width + 1

    # NaN gaps are displayed as white separators. They are not model parameters.
    grid = np.full((filter_count * tile_height - 1,
                    input_channels * tile_width - 1), np.nan)
    for filter_index in range(filter_count):
        for input_channel in range(input_channels):
            top = filter_index * tile_height
            left = input_channel * tile_width
            grid[top:top + kernel_height, left:left + kernel_width] = weights[filter_index, input_channel]

    color_limit = max(float(np.abs(weights).max()), 1e-8)
    color_map = plt.get_cmap('RdBu_r').copy()
    color_map.set_bad('white')
    fig, ax = plt.subplots(figsize=(max(6, input_channels * 0.35 + 2),
                                    max(5, filter_count * 0.25 + 2)), layout='constrained')
    picture = ax.imshow(grid, cmap=color_map, vmin=-color_limit, vmax=color_limit,
                        interpolation='nearest', aspect='auto')
    ax.set_xticks(np.arange(input_channels) * tile_width + (kernel_width - 1) / 2)
    ax.set_xticklabels(range(input_channels), fontsize=8)
    ax.set_yticks(np.arange(filter_count) * tile_height + (kernel_height - 1) / 2)
    ax.set_yticklabels(range(filter_count), fontsize=8)
    ax.set_xlabel('Input channel (each tile is one 3×3 kernel slice)')
    ax.set_ylabel('Output filter (one whole row is one complete filter)')
    ax.set_title(f'Block {block_number}: all {filter_count} filters × all {input_channels} input channels')
    fig.colorbar(picture, ax=ax, shrink=0.6, label='Signed weight value')
    fig.savefig(viz_dir / f'block_{block_number}_all_filter_slices.png', dpi=180)
    plt.show()
    print(f'Block {block_number}: displayed {filter_count} complete filters, '
          f'{filter_count * input_channels} slices, and {weights.size:,} weights (biases excluded).')

### Block 1: all 16 filters, each with one grayscale slice

Each row is one filter and there is a single column because the input has one channel. This is the same information as the 4×4 picture in step 17.

In [19]:
show_all_filter_slices(model.features[0], block_number=1)

Block 1: displayed 16 complete filters, 16 slices, and 144 weights (biases excluded).


### Block 2: all 32 filters, each with 16 learned-channel slices

Each row now has 16 tiles because this layer receives 16 pooled feature maps from block 1. All 16 slices contribute to one output feature map. They are not separate filters.

In [20]:
show_all_filter_slices(model.features[3], block_number=2)

Block 2: displayed 32 complete filters, 512 slices, and 4,608 weights (biases excluded).


### Block 3: all 64 filters, each with 32 learned-channel slices

This grid shows 64 complete filters. Each row contains 32 slices, one for each input channel from block 2. That is 2,048 individual 3×3 slices, but still **64 filters**, not 2,048 filters under our terminology.

In [21]:
show_all_filter_slices(model.features[6], block_number=3)

Block 3: displayed 64 complete filters, 2048 slices, and 18,432 weights (biases excluded).


## 18A. Enlarge one deep filter to see all its slices

The dense grid above shows everything at once. Here we select **block 3, filter 0** and enlarge its 32 input-channel slices. Together, these 32 panels are **one complete filter**. Change `filter_to_inspect` to any value from 0 to 63 to inspect a different filter in this block.

There is no unique way to collapse these 32 slices into a single image without discarding information or inventing a projection. Showing the slices explicitly is the faithful view of its actual weights.

In [22]:
filter_to_inspect = 0
deep_weights = model.features[6].weight.detach().cpu()
one_filter = deep_weights[filter_to_inspect]   # (32 input-channel slices, 3, 3)
deep_limit = max(deep_weights.abs().max().item(), 1e-8)
fig, axes = plt.subplots(4, 8, figsize=(13, 7), layout='constrained')

for channel_index, ax in enumerate(axes.flat):
    picture = ax.imshow(one_filter[channel_index], cmap='RdBu_r',
                        vmin=-deep_limit, vmax=deep_limit, interpolation='nearest')
    ax.set_title(f'Input {channel_index}', fontsize=9)
    ax.axis('off')

fig.colorbar(picture, ax=axes, shrink=0.65, label='Signed weight value')
fig.suptitle(f'ONE block-3 filter: filter {filter_to_inspect}, all 32 input-channel slices')
fig.savefig(viz_dir / 'one_deep_filter_all_channels.png', dpi=180)
plt.show()

# Confirm that creating these figures has not changed any learned weights or biases.
for name, value in model.state_dict().items():
    torch.testing.assert_close(value.cpu(), checkpoint['state_dict'][name], rtol=0, atol=0)
print('Verified: model weights and biases are unchanged.')

Verified: model weights and biases are unchanged.


## 19. Choose a digit image and prepare it as during training

The default is the first image of the MNIST test set; it is not chosen based on the model's prediction. You can change `image_index` to choose another test image.

The CNN learned from grayscale images scaled to 0–1. We must use the same preparation here: `ToTensor()` converts 0–255 pixels to 0–1. `unsqueeze(0)` adds a batch dimension, making the input `(1, 1, 28, 28)`.

In [23]:
image_index = 0   # Change this line to choose another test image.
input_tensor = test_dataset[image_index][0].unsqueeze(0)   # (1, 1, 28, 28), values 0 to 1
true_digit = int(test_dataset.targets[image_index])

print('Test image index:', image_index, '| true label:', true_digit)
print('Model input shape:', tuple(input_tensor.shape))
plt.figure(figsize=(4, 4))
plt.imshow(input_tensor[0, 0], cmap='gray')
plt.title('The digit given to the CNN')
plt.axis('off')
plt.tight_layout()
plt.savefig(viz_dir / 'input_digit.png', dpi=150)
plt.show()

Test image index: 0 | true label: 7
Model input shape: (1, 1, 28, 28)


## 20. Follow the image through the same model

Instead of creating multiple models or using hooks, we walk through the existing `model.features` layers in order. The weights are exactly the ones loaded earlier.

After each **ReLU**, we keep the current feature maps in a dictionary. These maps are nonnegative because ReLU replaces negative convolution outputs with zero. The following MaxPool shrinks the image before the next convolution.

The shape trace makes the progression visible. We then pass the last pooled result through the same classifier. As a check, the result must match calling `model(input_tensor)` normally. `no_grad()` avoids storing gradients; nothing is being trained.

In [24]:
feature_maps = {}
block_number = 0
model.eval()

with torch.no_grad():
    current = input_tensor
    for layer in model.features:
        if isinstance(layer, nn.Conv2d):
            block_number += 1
        current = layer(current)
        print(f'Block {block_number} | {layer.__class__.__name__:10s} -> {tuple(current.shape)}')
        if isinstance(layer, nn.ReLU):
            # Copy the map so the saved view stays independent of later operations.
            feature_maps[f'Block {block_number} — after ReLU'] = current.clone()

    logits = model.classifier(current)
    torch.testing.assert_close(logits, model(input_tensor))

print('Classifier output:', tuple(logits.shape))

Block 1 | Conv2d     -> (1, 16, 28, 28)
Block 1 | ReLU       -> (1, 16, 28, 28)
Block 1 | MaxPool2d  -> (1, 16, 14, 14)
Block 2 | Conv2d     -> (1, 32, 14, 14)
Block 2 | ReLU       -> (1, 32, 14, 14)
Block 2 | MaxPool2d  -> (1, 32, 7, 7)
Block 3 | Conv2d     -> (1, 64, 7, 7)
Block 3 | ReLU       -> (1, 64, 7, 7)
Block 3 | MaxPool2d  -> (1, 64, 3, 3)
Classifier output: (1, 10)


## 21. Display feature maps from all three convolutional blocks

One output channel gives one 2D feature map for this image. We display the first 16 channels of each block to keep the figures readable. The first block has 16 channels, the second 32, and the third 64; increase `maps_to_show` if you want to see more.

Within each figure, all panels share a color scale: dark indicates a small activation, and yellow indicates a larger activation. A completely dark map can be a valid result for this input. Each block uses its own scale, so brightness should not be compared numerically across different figures.

Feature maps are taken **before pooling** and have sizes 28×28, 14×14, and 7×7. Earlier maps often preserve more local detail. Deeper maps combine earlier features and cover larger image regions, but their meaning is not always visually obvious. A bright region is a response, not proof that the region caused the final prediction.

In [25]:
maps_to_show = 16

for block_index, (name, maps) in enumerate(feature_maps.items(), start=1):
    # Remove the batch dimension: (1, channels, height, width) -> (channels, height, width).
    maps = maps[0].cpu()
    count = min(maps_to_show, maps.shape[0])
    rows = (count + 3) // 4
    fig, axes = plt.subplots(rows, 4, figsize=(10, 2.6 * rows),
                             squeeze=False, layout='constrained')
    upper_limit = max(maps[:count].max().item(), 1e-8)

    for ax in axes.flat:
        ax.axis('off')
    for channel, ax in enumerate(axes.flat[:count]):
        image = ax.imshow(maps[channel], cmap='viridis', vmin=0, vmax=upper_limit)
        ax.set_title(f'Channel {channel}')

    fig.colorbar(image, ax=axes, shrink=0.7, label='Activation after ReLU')
    fig.suptitle(f'{name}: {maps.shape[0]} maps of {maps.shape[1]}×{maps.shape[2]} pixels (first {count} shown)')
    fig.savefig(viz_dir / f'block_{block_index}_feature_maps.png', dpi=150)
    plt.show()

## 22. Connect the maps to the final prediction

After the third pooling layer, Flatten combines the feature maps into a vector and the classifier produces one logit per digit. Softmax converts these scores into model probabilities for display. The largest score determines the predicted class.

This model was an educational baseline and can make mistakes. Its prediction and the visualizations should not be interpreted as a complete explanation of its decision.

In [26]:
probabilities = torch.softmax(logits, dim=1)[0]
predicted_index = probabilities.argmax().item()
print('Predicted digit:', class_names[predicted_index], '| True digit:', true_digit)

plt.figure(figsize=(7, 3))
plt.bar(range(len(class_names)), probabilities.numpy(), tick_label=list(range(len(class_names))))
plt.ylim(0, 1)
plt.xlabel('Digit')
plt.ylabel('Model probability')
plt.title('Prediction for the same digit image')
plt.tight_layout()
plt.savefig(viz_dir / 'prediction.png', dpi=150)
plt.show()

Predicted digit: 7 - seven | True digit: 7


## 23. A simple heatmap using Grad-CAM

**Which parts of this image support the model's predicted class?**

A **heatmap** displays numbers as colors. **Grad-CAM** is a method for creating a class-specific heatmap using feature maps and gradients. We reuse our trained CNN and the image from step 19. No new model, training, or image optimization is needed.

1. Get the last convolutional block's feature maps.
2. Calculate how the predicted class score depends on these maps (the gradients).
3. Use those gradients to weight the maps and combine them into one heatmap.
4. Enlarge the heatmap and place it over the image.

### A. Calculate the heatmap

Our last convolution is `features[6]`, followed by ReLU at `[7]` and pooling at `[8]`. We use the maps **after that ReLU, before pooling**. For MNIST these are 64 maps of 7×7. Each map gets one importance weight: its average gradient for the selected class score.

We use the raw class score (a logit), not a softmax probability. Gradients are needed here, so do **not** wrap this calculation in `torch.no_grad()`. `model.eval()` is still appropriate: evaluation mode does not disable gradients.

In [27]:
model.eval()

# A separate input lets PyTorch track this calculation, even if weights are frozen.
# Neither this image nor any model weights will be updated.
cam_input = input_tensor.detach().clone().requires_grad_(True)

# Run the SAME CNN, stopping briefly at the final block's feature maps.
maps = model.features[:8](cam_input)       # Last convolution + ReLU: [1, 64, 7, 7].
pooled_maps = model.features[8](maps)      # The remaining max-pooling layer.
scores = model.classifier(pooled_maps)     # One raw score per digit class.
class_index = scores[0].argmax().item()    # Explain the predicted class.

# How sensitive is this class score to each value in the feature maps?
gradients = torch.autograd.grad(scores[0, class_index], maps)[0]

# Average over height and width: one importance weight per feature map.
weights = gradients.mean(dim=(2, 3), keepdim=True)

# Combine the 64 weighted maps, then keep only positive contributions.
heatmap = (weights * maps).sum(dim=1, keepdim=True)
heatmap = heatmap.relu().detach()

# Enlarge the small map to match the image and scale its values to 0–1.
heatmap = F.interpolate(
    heatmap, size=cam_input.shape[-2:], mode='bilinear', align_corners=False
)[0, 0]
heatmap = heatmap / heatmap.max().clamp(min=1e-8)   # Avoid division by zero.

print('Explaining the prediction:', class_names[class_index])
if heatmap.max().item() == 0:
    print('This example has no positive Grad-CAM response; try another image.')

Explaining the prediction: 7 - seven


### B. Show the image, heatmap, and overlay

Brighter yellow regions indicate stronger positive contributions **according to Grad-CAM** for the named class. Dark regions have low or no positive contribution in this map.

This is a coarse explanation (it starts from a 7×7 map), not an exact stroke outline or proof of what the model understands. The 0–1 scale is relative within this image; it is **not a probability**. Unlike an individual feature map, Grad-CAM combines multiple maps to explain a particular class score.

In [28]:
digit_image = input_tensor[0, 0].detach().cpu().numpy()
heatmap_image = heatmap.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(11, 3))

axes[0].imshow(digit_image, cmap='gray')
axes[0].set_title('Original digit')

color_plot = axes[1].imshow(heatmap_image, cmap='inferno', vmin=0, vmax=1)
axes[1].set_title('Grad-CAM heatmap')
fig.colorbar(color_plot, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(digit_image, cmap='gray')
# alpha controls transparency so the original digit remains visible.
axes[2].imshow(heatmap_image, cmap='inferno', vmin=0, vmax=1, alpha=0.45)
axes[2].set_title(f'Overlay: {class_names[class_index]}')

for axis in axes:
    axis.axis('off')
fig.tight_layout()
fig.savefig(viz_dir / 'simple_gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved visualization images to:', viz_dir.resolve())

Saved visualization images to: /tmp/claude-0/-home-user-lec03/82af13de-b088-581a-a30e-c84accc06e02/scratchpad/run/outputs/mnist/filter_visualizations


## Summary : How the pieces fit together

| What we did | What it means |
|---|---|
| Torchvision `MNIST` Dataset | Pairs every 28×28 image with its label; `DataLoader` batches it directly. |
| Train / validation / test | Training teaches the weights, validation selects the epoch, the 10,000 official test images are used once at the end. |
| Vanilla network (Part A) | Flattens each image into 784 numbers; ignores pixel neighbors. |
| CNN (Part B) | Reuses small filters across the image; evaluated with accuracy, per-class metrics, confusion matrix, and mistakes. |
| Filter weights | Learned numbers that stay fixed when the input changes. |
| Feature maps | Responses of individual channels to an input image. |
| Class scores | The classifier's outputs for the ten digit categories. |
| Grad-CAM heatmap | A combination of feature maps highlighting regions that support a selected class score. |